# Fase C — Backtesting y evaluacion de los 4 modelos

Tesis MCD USACH · Portafolio gestionado por IA.

Este notebook ejecuta el backtest **walk-forward** (sin look-ahead) definido en
`app/pipeline/backtest.py` y resume las metricas que sustentan la defensa:

- **LSTM / Prophet** (regresion de precio/tendencia): RMSE, MAE, MAPE, precision direccional.
- **XGBoost / Random Forest** (clasificacion de senal/riesgo): accuracy, F1 macro.
- **Estrategia** long-only guiada por la senal de XGBoost vs. buy&hold del universo
  y de los indices de contexto (IPSA proxy ECH, Dow Jones).

> Ejecutar el kernel del venv `backend/.venv` (Python 3.12). En Windows: `set PYTHONUTF8=1`.

In [ ]:
import os, sys
os.environ.setdefault('PYTHONUTF8', '1')
sys.path.insert(0, os.path.abspath('..'))  # raiz del backend

import pandas as pd
import matplotlib.pyplot as plt

from app.constants import UNIVERSE
from app.data.yahoo import get_history
from app.pipeline import backtest as bt

pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
print('Universo:', UNIVERSE)

## 1. Descarga de historicos (5y, Yahoo)

In [ ]:
histories = {}
for s in UNIVERSE:
    try:
        histories[s] = get_history(s, period='5y')
    except Exception as e:
        print('!', s, e)
{s: df.shape for s, df in histories.items()}

## 2. Backtest walk-forward de los 4 modelos

`retrain_folds=4` reentrena con ventana expansiva en 4 cortes temporales.
Cada prediccion usa solo datos previos (igual que produccion).

In [ ]:
rows = bt.backtest_models(histories, list(bt.BT_CONFIG), retrain_folds=4)
df_m = pd.DataFrame(rows)
df_m.head(40)

### Metricas globales por modelo (symbol nulo = agregado)

In [ ]:
glob = df_m[df_m['symbol'].isna()].pivot_table(index='model', columns='metric', values='value')
glob

In [ ]:
# Precision direccional (regresion) y accuracy (clasificacion) por modelo.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for col, ax, title in [('dir_acc', axes[0], 'Precision direccional (LSTM/Prophet)'),
                       ('accuracy', axes[1], 'Accuracy (XGBoost/RandomForest)')]:
    if col in glob:
        glob[col].dropna().plot.bar(ax=ax, color='#2563eb')
        ax.axhline(50, ls='--', c='gray'); ax.set_title(title); ax.set_ylabel('%')
plt.tight_layout(); plt.show()

## 3. Backtest de estrategia vs. benchmarks

In [ ]:
srows = bt.strategy_backtest(histories, retrain_folds=4)
df_s = pd.DataFrame(srows)
df_s.pivot_table(index='model', columns='metric', values='value')

## 4. (Opcional) Persistir metricas en Supabase (`model_metrics`)

Requiere `SUPABASE_URL` y `SUPABASE_SERVICE_ROLE_KEY` en `backend/.env`.
Equivale a `python -m app.pipeline.backtest --write`.

In [ ]:
# from app.db.supabase_client import insert_metrics
# insert_metrics(rows + srows)
# print('escritas', len(rows) + len(srows), 'filas')

## Notas para el comite

- Walk-forward = sin fuga de informacion: el modelo nunca ve el futuro al predecir.
- IPSA via Yahoo (`^IPSA`) se descontinuo en 2019 → se usa el ETF **ECH** como proxy.
- Pendiente (mejora): AUC ROC multiclase requiere exponer probabilidades en `predict_one`.